In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

## Loading Dataset ##

In [2]:
df=pd.read_csv("/Users/deepanshus/StockMarketPredictor/Project/data/raw/Banks/UNBK Historical Data.csv")
df.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,13-03-2026,173.88,181.50,181.64,173.31,10.94M,-4.51%
1,12-03-2026,182.10,179.52,183.40,175.52,14.23M,0.65%
2,11-03-2026,180.92,187.50,187.55,180.40,12.58M,-2.89%
3,10-03-2026,186.30,182.00,187.55,179.80,18.49M,4.04%
4,09-03-2026,179.06,181.51,183.16,175.50,22.08M,-5.08%


## Preprocessing ##

In [3]:
df =df.rename(columns={
    "Price":"close",
    "Open":"open",
    "High":"high",
    "Low":"low",
    "Vol.":"volume",
    "Change %":"change_percent"

})
df["date"] = pd.to_datetime(df["Date"])

df = df.sort_values("date")
df = df.reset_index(drop=True)

/var/folders/g8/43rm_8rs5wbchfhhkxt_832c0000gn/T/ipykernel_45185/349176893.py:10: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df["Date"])


In [4]:
# # chaning volume

df.head()
df.isna().sum()

Date              0
close             0
open              0
high              0
low               0
volume            0
change_percent    0
date              0
dtype: int64

In [5]:
df[df.isnull().any(axis=1)]

,Date,close,open,high,low,volume,change_percent,date


In [6]:
def convert_volume(x):
    
    if isinstance(x, str):

        if "M" in x:
            return float(x.replace("M","")) * 1_000_000

        elif "K" in x:
            return float(x.replace("K","")) * 1_000

        else:
            return float(x)

    return x

df["volume"]=df["volume"].apply(convert_volume)

In [7]:
df['change_percent']=df['change_percent'].str.replace('%',"").astype(float)


In [8]:
# cols = ["open","high","low","close"]

# for c in cols:
#     df[c] = pd.to_numeric(df[c])]

cols = ["open", "high", "low", "close"]

for c in cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

## Calculating Indicators and making Dataset

In [9]:
# creating functions for calculating features
def sort_dataset(df):

    df = df.copy()
    df = df.sort_values("date").reset_index(drop=True)

    return df
def calculate_returns(df):

    df["return"] = df["close"].pct_change()

    return df
def calculate_volume_change(df):

    df["volume_change"] = df["volume"].pct_change()

    return df
def calculate_volatility(df):

    df["volatility"] = df["return"].rolling(10).std()

    return df
def calculate_sma(df):

    df["sma10"] = df["close"].rolling(10).mean()

    df["sma50"] = df["close"].rolling(50).mean()

    df["sma_ratio"] = df["sma10"] / df["sma50"]

    return df
def calculate_macd(df):

    ema12 = df["close"].ewm(span=12, adjust=False).mean()

    ema26 = df["close"].ewm(span=26, adjust=False).mean()

    df["macd"] = ema12 - ema26

    return df
def calculate_rsi(df):

    delta = df["close"].diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()

    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["rsi"] = 100 - (100 / (1 + rs))

    return df
def create_target(df):

    df["target"] = df["close"].shift(-1) / df["close"] - 1

    return df
def clean_dataset(df):

    df = df.dropna()

    return df

In [10]:
# calling functions
df = sort_dataset(df)

df = calculate_returns(df)

df = calculate_volume_change(df)

df = calculate_volatility(df)

df = calculate_sma(df)

df = calculate_macd(df)

df = calculate_rsi(df)

df = create_target(df)

df = clean_dataset(df)

indicator_df = df[[
    "rsi",
    "macd",
    "sma_ratio",
    "volume_change",
    "volatility",
    "target"
]]
indicator_df.to_csv("indicator_dataset.csv", index=False)